## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [2]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [3]:
MY_BUBBLE_FILE = "conspirationist.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: conspirationist
Texte: 50


,id,agent,text
0,yt_joXkZDqGZQU_UgzFU0NMeTYppU_dHpd4AaABAg,Conspiraționist,Dar de românii din Ucraina care și-au pierdut ...
1,yt_X3bwh1-9nUU_Ugzkb5UdoDsVooo_b2V4AaABAg,Conspiraționist,"Ținând cont de microfoane, unde sunt microfoan..."
2,yt_M37Lar0c11g_UgxQUM1Vz9INIGxd0TJ4AaABAg,Conspiraționist,Aveti incredere in cine.a juns sa fie in CSAT ...
3,yt_6_Hc2S02Duw_UgwRkTZLRtfW1ljXoWl4AaABAg,Conspiraționist,Ce aveti bă cu bietul om? Tot nu intelegeti ca...
4,yt_qkGhsJFft00_Ugx92UxyhOpyG68UR1l4AaABAg,Conspiraționist,"Pandemia, a fost doar testul care a evidentiat..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [4]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Dar de românii din Ucraina care și-au pierdut dreptul de a învăța în școli românești ați discutat? De ce nu ați discutat și despre preoții ortodocși români care au fost agresați de acest domn Zelinsky? Dar despre cum își recrutează domnul Zelinsky soldații,trimițându-i la moarte sigură? Despre spăgile pe care vameșii ucrainieni le cereau femeilor și copiilor să părăsească țara? Epstein files? Nu? Pedofilia la care a fost expus dumnealui cu domnul Trump nu? Ați omis? Mă gândeam eu!


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [5]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\Carmen Ciutu\Desktop\proiect AI\echochamber-project-team-4\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Carmen Ciutu\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 2/

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 50 de texte.
# Au fost generați 50 de vectori.
# A doua valoare din embeddings.shape reprezintă 384 de dimensiuni ale vectorilor (pentru modelul ales).

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [6]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\conspirationist
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: da
# index.pkl există: da
# index.ntotal este egal cu numărul de texte: da

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [13]:
# Text nou introdus în aplicație

input_text = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."

In [14]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [15]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.543
Text: Dacă motivele pentru anularea alegerilor sunt cele menționate de dumneavoastră doamna Simina atunci trebuia să anulați alegerile și în Mai 2025 luând în considerare modul de finanțare a campaniei lui Nicușor Dan și promovarea campaniei pe TikTok care a fost mult mai agresivă decât a lui Călin Georgescu . De ce nu ați anulat alegerile prezidențiale de ce nu ați făcut-o ? NU AȚI PRIMIT ORDIN PE UNITATE ? Mai am o singură întrebare pentru dumneavoastră doamna Simina: DORMIȚI BINE ??????

Rezultat 2
Scor: 0.428
Text: KGB ul ukr a avut un rol în anularea alegerilor , ii au la mana , au semnat niste ilegitimi trădători

Rezultat 3
Scor: 0.384
Text: Ținând cont de microfoane, unde sunt microfoanele PROTV, Antenele, TRV 1 s.a.? Desecretizarea, turul 2 înapoi, vreau sa votez liber, daca acest fapt împlinit, dus la lovitura de stat din decembrie 2024 nu este principal subiect intr.o tara democratica atunci CORUPTIA si PROSTIA ucide.

Rezultat 4
Scor: 0.286
Text: Ave

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

In [18]:
# VERIFICARE - Afirmație politică cu vocea conspiraționist
input_text = "Dovezi arată că interferența electorală externă a fost ignorată deliberat de autoritățile centrale"
print("Input verificare:", input_text)

Input verificare: Dovezi arată că interferența electorală externă a fost ignorată deliberat de autoritățile centrale


In [19]:
# Transformare în embedding și căutare
query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.469
Text: Dacă motivele pentru anularea alegerilor sunt cele menționate de dumneavoastră doamna Simina atunci trebuia să anulați alegerile și în Mai 2025 luând în considerare modul de finanțare a campaniei lui Nicușor Dan și promovarea campaniei pe TikTok care a fost mult mai agresivă decât a lui Călin Georgescu . De ce nu ați anulat alegerile prezidențiale de ce nu ați făcut-o ? NU AȚI PRIMIT ORDIN PE UNITATE ? Mai am o singură întrebare pentru dumneavoastră doamna Simina: DORMIȚI BINE ??????

Rezultat 2
Scor: 0.466
Text: KGB ul ukr a avut un rol în anularea alegerilor , ii au la mana , au semnat niste ilegitimi trădători

Rezultat 3
Scor: 0.459
Text: Ținând cont de microfoane, unde sunt microfoanele PROTV, Antenele, TRV 1 s.a.? Desecretizarea, turul 2 înapoi, vreau sa votez liber, daca acest fapt împlinit, dus la lovitura de stat din decembrie 2024 nu este principal subiect intr.o tara democratica atunci CORUPTIA si PROSTIA ucide.

Rezultat 4
Scor: 0.364
Text: Pen

Toate rezultatele exprima vocea agentlui, in afara de ultimul pe care nu il vad 100% potrivit
